# TTFS-SNN varying width-depth regions 

In [4]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from ttfs_regions.utils import resolve_device
DEVICE = resolve_device("auto")
print("repo:", ROOT)
print("device:", DEVICE)


In [5]:
from ttfs_regions.configs import WidthDepthConfig
from ttfs_regions.runners import run_width_depth
from ttfs_regions.plotting import plot_width_depth

DATASET = "cifar10"          # "mnist" or "cifar10"
FAST = False                # set False for paper-scale run
DATA_ROOT = ROOT / "data"
RESULTS_ROOT = ROOT / "results"


In [6]:
if FAST:
    cfg = WidthDepthConfig(
        steps=100, seeds=(42,), num_pairs=2, calibration_size=16,
        widths=(8, 16), schedules={"8": [8, 8], "16": [16, 16]},
        ttfs_batch_size=16, delays_on=False,
    )
else:
    cfg = WidthDepthConfig()

runs = {}
for init_name in ("init1", "init2"):
    runs[init_name] = run_width_depth(
        DATASET, "ttfs", DATA_ROOT, RESULTS_ROOT,
        init_name=init_name, config=cfg, device=DEVICE,
    )
    display(runs[init_name]["width_summary"])
    plot_width_depth(
        runs[init_name]["width_summary"],
        runs[init_name]["depth_summary"],
        cfg.schedules,
        RESULTS_ROOT / f"{DATASET}_ttfs_{init_name}_notebook.png",
    )
